# Syllable use per trial mode (supplementary)

fig_segmentation panel K — the syllable composition over the time-warped trial, one stacked column per bin (paw states pale/still → dark/vigorous; whisking and licking hatched) — drawn for the trials of **each of the 17 trial modes**. Each mode keeps its number and colour from fig_engagement_syllables (numbered 1..17 in colour-wheel order of the UMAP map, `umap_pastel` palette), shown in the swatch above its panel with its share of all trials. Trials: every trial of the mode fit `3_trial_modes/trial_modes_8_k_10_bin_syllables_02-10-2026` (all proficient sessions); its epoch columns already hold the syllable codes in the uniform paw numbering. Same drawing function as fig_segmentation K (`paper_style.syllable_histogram`). Companion to `fig_trial_mode_curves` (same mode order and swatches).

In [ ]:
import sys, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from datetime import date

FIGDIR = pathlib.Path.cwd().resolve()
while not (FIGDIR / 'fig_layout.py').exists() and FIGDIR != FIGDIR.parent:
    FIGDIR = FIGDIR / 'figure_generation' if (FIGDIR / 'figure_generation').exists() else FIGDIR.parent
PI = FIGDIR.parent
REPO = PI.parent
for p in [str(REPO), str(PI), str(FIGDIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)
from Functions import trial_modes as tmodes
import paper_style as ps
import fig_layout as fl
ps.use('paper')
ps.use_paw_states('uniform')
FIG = 'fig_trial_mode_syllables'
MODES_FILE = PI / '3_trial_modes' / 'trial_modes_8_k_10_bin_syllables_02-10-2026' / 'trials.pqt'
TEXTURE = True
N_COLS = 6

## Trials, modes and their colours

In [ ]:
trials = pd.read_parquet(MODES_FILE)
for e in tmodes.EPOCHS:
    trials[e] = trials[e].map(np.asarray)
ORDER = [int(k) for k in tmodes.umap_order(trials)]              # cluster ids in colour-wheel order
MODE_NAME = {k: i + 1 for i, k in enumerate(ORDER)}              # cluster id -> displayed number (as fig_engagement_syllables)
MODE_COLORS = tmodes.umap_colors(trials, lightness=76, chroma=40)    # the 'umap_pastel' palette
ink_on = lambda c: 'white' if ps.hex_to_oklch(c)[0] < 0.62 else 'k'
_, SEQS = tmodes.trial_matrix_from_trials(trials)                # trials x 40 codes, rows aligned with `trials`
cl = trials['trial_cluster'].to_numpy()
CODES = {k: SEQS[cl == k] for k in ORDER}
SHARE = {k: len(CODES[k]) / np.isfinite(cl).sum() for k in ORDER}
print(f"{len(trials):,} trials, {trials['session'].nunique()} sessions; per mode: "
      + ', '.join(f'{MODE_NAME[k]}: {len(CODES[k]):,}' for k in ORDER))

## The figure

In [ ]:
n_rows = int(np.ceil((len(ORDER) + 1) / N_COLS))
fig = plt.figure(figsize=(fl.FIG_WIDTH, 1.42 * n_rows + 0.2))
G = fig.add_gridspec(n_rows, N_COLS, wspace=0.14, hspace=0.38, left=0.06, right=0.99, top=0.975, bottom=0.08)
for i, k in enumerate(ORDER):
    r_, c_ = divmod(i, N_COLS)
    g = G[r_, c_].subgridspec(2, 1, height_ratios=[0.16, 1], hspace=0.06)
    sw = fig.add_subplot(g[0]); sw.set_axis_off()
    col = MODE_COLORS[float(k)]
    sw.add_patch(Rectangle((0, 0), 1, 1, transform=sw.transAxes, color=col, lw=0))
    sw.text(0.5, 0.5, f"mode {MODE_NAME[k]}   {100 * SHARE[k]:.1f}%", transform=sw.transAxes, ha='center',
            va='center', color=ink_on(col), fontsize=plt.rcParams['font.size'] * 0.8, fontweight='bold')
    ax = fig.add_subplot(g[1])
    ps.syllable_histogram(CODES[k], ax=ax, texture=TEXTURE, warn=False)
    ps.epoch_lines(ax, CODES[k].shape[1])
    ax.set_ylabel('')
    ax.tick_params(length=1.5, labelsize=plt.rcParams['xtick.labelsize'] * 0.75)
    if c_ == 0:
        ax.set_yticks([0, 1]); ax.set_ylabel('Fraction', fontsize=plt.rcParams['font.size'] * 0.75, labelpad=0)
    else:
        ax.set_yticks([])
    if i + N_COLS >= len(ORDER):                                    # nothing below it
        ax.set_xticks([5, 15, 25, 35]); ax.set_xticklabels(ps.EPOCH_SHORT, rotation=45, ha='right')
    else:
        ax.set_xticks([])
# the free slot: the paw-state and whisk / lick key
ax = fig.add_subplot(G[divmod(len(ORDER), N_COLS)]); ax.set_axis_off()
ps.paw_legend(ax, ncol=1, hatch=TEXTURE, loc='center', bbox_to_anchor=(0.5, 0.5),
              fontsize=plt.rcParams['legend.fontsize'] * 0.7)
plt.show()
OUT = FIGDIR / 'figures'
for path in [OUT / f"{FIG}_{date.today().strftime('%d-%m-%Y')}.svg", OUT / f'{FIG}.pdf', OUT / f'{FIG}.png']:
    fig.savefig(path, dpi=300)
    print('wrote', path.relative_to(PI))